# Feature Engineering — Multi-table Aggregation

**Week 3: Feature Engineering (Multi-table Aggregation)**

Aggregating 5 Home Credit secondary tables to the `SK_ID_CURR` level so they can be merged into `application_train`:
- `bureau` — credit history from other bureaus/institutions
- `previous_application` — history of previous applications to Home Credit
- `installments_payments` — installment payment history
- `POS_CASH_balance` — POS/cash loan history
- `credit_card_balance` — credit card history

For each table: brief exploration → build an aggregation function → check the new features' correlation with `TARGET` → save. At the end, everything is merged into a single **master feature table**.


## 1. Bureau — Credit History from Other Bureaus

### 1.1 Load & Explore


In [1]:
import pandas as pd
import numpy as np

bureau = pd.read_csv('../data/raw/bureau.csv')
print(bureau.shape)
bureau.head()

(1716428, 17)


,SK_ID_CURR,SK_ID_BUREAU,CREDIT_ACTIVE,CREDIT_CURRENCY,DAYS_CREDIT,CREDIT_DAY_OVERDUE,DAYS_CREDIT_ENDDATE,DAYS_ENDDATE_FACT,AMT_CREDIT_MAX_OVERDUE,CNT_CREDIT_PROLONG,AMT_CREDIT_SUM,AMT_CREDIT_SUM_DEBT,AMT_CREDIT_SUM_LIMIT,AMT_CREDIT_SUM_OVERDUE,CREDIT_TYPE,DAYS_CREDIT_UPDATE,AMT_ANNUITY
0,215354,5714462,Closed,currency 1,-497,0,-153.0,-153.0,NaN,0,91323.0,0.0,NaN,0.0,Consumer credit,-131,NaN
1,215354,5714463,Active,currency 1,-208,0,1075.0,NaN,NaN,0,225000.0,171342.0,NaN,0.0,Credit card,-20,NaN
2,215354,5714464,Active,currency 1,-203,0,528.0,NaN,NaN,0,464323.5,NaN,NaN,0.0,Consumer credit,-16,NaN
3,215354,5714465,Active,currency 1,-203,0,NaN,NaN,NaN,0,90000.0,NaN,NaN,0.0,Credit card,-16,NaN
4,215354,5714466,Active,currency 1,-629,0,1197.0,NaN,77674.5,0,2700000.0,NaN,NaN,0.0,Consumer credit,-21,NaN


In [2]:
bureau.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1716428 entries, 0 to 1716427
Data columns (total 17 columns):
 #   Column                  Dtype  
---  ------                  -----  
 0   SK_ID_CURR              int64  
 1   SK_ID_BUREAU            int64  
 2   CREDIT_ACTIVE           object 
 3   CREDIT_CURRENCY         object 
 4   DAYS_CREDIT             int64  
 5   CREDIT_DAY_OVERDUE      int64  
 6   DAYS_CREDIT_ENDDATE     float64
 7   DAYS_ENDDATE_FACT       float64
 8   AMT_CREDIT_MAX_OVERDUE  float64
 9   CNT_CREDIT_PROLONG      int64  
 10  AMT_CREDIT_SUM          float64
 11  AMT_CREDIT_SUM_DEBT     float64
 12  AMT_CREDIT_SUM_LIMIT    float64
 13  AMT_CREDIT_SUM_OVERDUE  float64
 14  CREDIT_TYPE             object 
 15  DAYS_CREDIT_UPDATE      int64  
 16  AMT_ANNUITY             float64
dtypes: float64(8), int64(6), object(3)
memory usage: 222.6+ MB


In [3]:
bureau['CREDIT_ACTIVE'].value_counts()

CREDIT_ACTIVE
Closed      1079273
Active       630607
Sold           6527
Bad debt         21
Name: count, dtype: int64

**Insight:** most credits are already `Closed` (1,079,273), followed by `Active` (630,607). `Sold` and `Bad debt` have small counts but are still worth flagging since they could be predictive.


In [4]:
bureau.groupby('SK_ID_CURR').size().describe()

count    305811.000000
mean          5.612709
std           4.430354
min           1.000000
25%           2.000000
50%           4.000000
75%           8.000000
max         116.000000
dtype: float64

**Insight:** the median customer has 4 loans recorded in the bureau, but there's a long tail up to 116 loans — this variance is why count/ratio features per customer are more informative than raw rows.


### 1.2 Feature Engineering

Build flags for problematic conditions (`FLAG_OVERDUE`, `FLAG_BAD_DEBT`), then aggregate to the `SK_ID_CURR` level. Besides raw counts, **ratio** versions are deliberately added (`BUREAU_RATIO_ACTIVE`, `BUREAU_DEBT_CREDIT_RATIO`, `BUREAU_RATIO_OVERDUE`) — proportions are usually fairer than raw counts since they aren't biased toward customers with a lot of credit history.


In [5]:
def engineer_bureau_features(bureau):
    bureau = bureau.copy()

    bureau['FLAG_OVERDUE'] = (bureau['CREDIT_DAY_OVERDUE'] > 0).astype(int)
    bureau['FLAG_BAD_DEBT'] = (bureau['CREDIT_ACTIVE'] == 'Bad debt').astype(int)

    agg = bureau.groupby('SK_ID_CURR').agg(
        BUREAU_COUNT_LOANS=('SK_ID_BUREAU', 'count'),
        BUREAU_COUNT_ACTIVE=('CREDIT_ACTIVE', lambda x: (x == 'Active').sum()),
        BUREAU_COUNT_CLOSED=('CREDIT_ACTIVE', lambda x: (x == 'Closed').sum()),
        BUREAU_COUNT_BAD_DEBT=('FLAG_BAD_DEBT', 'sum'),

        BUREAU_SUM_OVERDUE_COUNT=('FLAG_OVERDUE', 'sum'),
        BUREAU_MAX_DAYS_OVERDUE=('CREDIT_DAY_OVERDUE', 'max'),
        BUREAU_MEAN_DAYS_OVERDUE=('CREDIT_DAY_OVERDUE', 'mean'),

        BUREAU_TOTAL_CREDIT_SUM=('AMT_CREDIT_SUM', 'sum'),
        BUREAU_TOTAL_CREDIT_DEBT=('AMT_CREDIT_SUM_DEBT', 'sum'),
        BUREAU_TOTAL_CREDIT_OVERDUE=('AMT_CREDIT_SUM_OVERDUE', 'sum'),
        BUREAU_MAX_CREDIT_OVERDUE=('AMT_CREDIT_MAX_OVERDUE', 'max'),

        BUREAU_MEAN_CNT_PROLONG=('CNT_CREDIT_PROLONG', 'mean'),
        BUREAU_MAX_CNT_PROLONG=('CNT_CREDIT_PROLONG', 'max'),

        BUREAU_MEAN_DAYS_CREDIT=('DAYS_CREDIT', 'mean'),
    ).reset_index()

    agg['BUREAU_RATIO_ACTIVE'] = agg['BUREAU_COUNT_ACTIVE'] / agg['BUREAU_COUNT_LOANS']
    agg['BUREAU_DEBT_CREDIT_RATIO'] = agg['BUREAU_TOTAL_CREDIT_DEBT'] / agg['BUREAU_TOTAL_CREDIT_SUM']
    agg['BUREAU_DEBT_CREDIT_RATIO'] = agg['BUREAU_DEBT_CREDIT_RATIO'].replace([np.inf, -np.inf], np.nan)

    # new feature: proportion of loans that were ever overdue (fairer than raw count)
    agg['BUREAU_RATIO_OVERDUE'] = agg['BUREAU_SUM_OVERDUE_COUNT'] / agg['BUREAU_COUNT_LOANS']

    return agg

### 1.3 Correlation with Target


In [6]:
bureau_features = engineer_bureau_features(bureau)

df_check = pd.read_csv('../data/processed/application_train_with_anomaly.csv')
df_check = df_check[['SK_ID_CURR', 'TARGET']].merge(bureau_features, on='SK_ID_CURR', how='left')

correlations = df_check.drop(columns=['SK_ID_CURR']).corr()['TARGET'].sort_values()
print(correlations)


--- Bureau Feature Correlation Results (Latest) ---
BUREAU_COUNT_CLOSED           -0.030812
BUREAU_TOTAL_CREDIT_SUM       -0.014057
BUREAU_MAX_CREDIT_OVERDUE      0.002540
BUREAU_MEAN_CNT_PROLONG        0.003031
BUREAU_MAX_CNT_PROLONG         0.003951
BUREAU_COUNT_BAD_DEBT          0.004003
BUREAU_COUNT_LOANS             0.004056
BUREAU_MAX_DAYS_OVERDUE        0.005493
BUREAU_TOTAL_CREDIT_DEBT       0.007144
BUREAU_MEAN_DAYS_OVERDUE       0.008118
BUREAU_TOTAL_CREDIT_OVERDUE    0.013335
BUREAU_RATIO_OVERDUE           0.031932
BUREAU_SUM_OVERDUE_COUNT       0.042007
BUREAU_DEBT_CREDIT_RATIO       0.060235
BUREAU_COUNT_ACTIVE            0.067128
BUREAU_RATIO_ACTIVE            0.077356
BUREAU_MEAN_DAYS_CREDIT        0.089729
TARGET                         1.000000
Name: TARGET, dtype: float64


**Insight:** `BUREAU_MEAN_DAYS_CREDIT` (0.090) and `BUREAU_RATIO_ACTIVE` (0.077) turn out to be the strongest bureau features — both are ratios/averages, not raw counts. This pattern will show up consistently again in the other tables.


### 1.4 Save


In [7]:
bureau_features.to_csv('../data/processed/bureau_features.csv', index=False)

## 2. Previous Application — History of Previous Applications

### 2.1 Load & Explore


In [8]:
prev_app = pd.read_csv('../data/raw/previous_application.csv')
print(prev_app.shape)
prev_app.head()

(1670214, 37)


,SK_ID_PREV,SK_ID_CURR,NAME_CONTRACT_TYPE,AMT_ANNUITY,AMT_APPLICATION,AMT_CREDIT,AMT_DOWN_PAYMENT,AMT_GOODS_PRICE,WEEKDAY_APPR_PROCESS_START,HOUR_APPR_PROCESS_START,...,NAME_SELLER_INDUSTRY,CNT_PAYMENT,NAME_YIELD_GROUP,PRODUCT_COMBINATION,DAYS_FIRST_DRAWING,DAYS_FIRST_DUE,DAYS_LAST_DUE_1ST_VERSION,DAYS_LAST_DUE,DAYS_TERMINATION,NFLAG_INSURED_ON_APPROVAL
0,2030495,271877,Consumer loans,1730.430,17145.0,17145.0,0.0,17145.0,SATURDAY,15,...,Connectivity,12.0,middle,POS mobile with interest,365243.0,-42.0,300.0,-42.0,-37.0,0.0
1,2802425,108129,Cash loans,25188.615,607500.0,679671.0,NaN,607500.0,THURSDAY,11,...,XNA,36.0,low_action,Cash X-Sell: low,365243.0,-134.0,916.0,365243.0,365243.0,1.0
2,2523466,122040,Cash loans,15060.735,112500.0,136444.5,NaN,112500.0,TUESDAY,11,...,XNA,12.0,high,Cash X-Sell: high,365243.0,-271.0,59.0,365243.0,365243.0,1.0
3,2819243,176158,Cash loans,47041.335,450000.0,470790.0,NaN,450000.0,MONDAY,7,...,XNA,12.0,middle,Cash X-Sell: middle,365243.0,-482.0,-152.0,-182.0,-177.0,1.0
4,1784265,202054,Cash loans,31924.395,337500.0,404055.0,NaN,337500.0,THURSDAY,9,...,XNA,24.0,high,Cash Street: high,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
print(prev_app['NAME_CONTRACT_STATUS'].value_counts())
print()
print(prev_app.groupby('SK_ID_CURR').size().describe())

NAME_CONTRACT_STATUS
Approved        1036781
Canceled         316319
Refused          290678
Unused offer      26436
Name: count, dtype: int64

count    338857.000000
mean          4.928964
std           4.220716
min           1.000000
25%           2.000000
50%           4.000000
75%           7.000000
max          77.000000
dtype: float64


**Insight:** out of 1.67 million previous application rows, ~16% have a `Refused` status at the row level. The median customer has 4 previous applications (some up to 77) — this rejection history is a strong candidate predictor.


### 2.2 Feature Engineering

Just like `DAYS_EMPLOYED` in notebook 1, the `DAYS_*` columns here also have a `365243` placeholder that needs to be treated as NaN. Main features: approved/refused ratio, and the `AMT_CREDIT` vs `AMT_APPLICATION` ratio (an indicator of whether the customer got the amount they requested or a reduced amount).


In [10]:
def engineer_previous_application_features(prev_app):
    prev_app = prev_app.copy()
    
    # Treat the 365243 placeholder (same as DAYS_EMPLOYED in application_train)
    days_cols = ['DAYS_FIRST_DRAWING', 'DAYS_FIRST_DUE', 'DAYS_LAST_DUE_1ST_VERSION', 
                 'DAYS_LAST_DUE', 'DAYS_TERMINATION']
    for col in days_cols:
        prev_app[col] = prev_app[col].replace(365243, np.nan)
    
    # Contract status flags
    prev_app['FLAG_APPROVED'] = (prev_app['NAME_CONTRACT_STATUS'] == 'Approved').astype(int)
    prev_app['FLAG_REFUSED'] = (prev_app['NAME_CONTRACT_STATUS'] == 'Refused').astype(int)
    
    # Approved amount vs applied amount ratio (indicates whether the customer got what they requested or less)
    prev_app['APPLICATION_CREDIT_RATIO'] = prev_app['AMT_CREDIT'] / prev_app['AMT_APPLICATION']
    prev_app['APPLICATION_CREDIT_RATIO'] = prev_app['APPLICATION_CREDIT_RATIO'].replace([np.inf, -np.inf], np.nan)
    
    agg = prev_app.groupby('SK_ID_CURR').agg(
        PREV_COUNT_APPLICATIONS=('SK_ID_PREV', 'count'),
        PREV_COUNT_APPROVED=('FLAG_APPROVED', 'sum'),
        PREV_COUNT_REFUSED=('FLAG_REFUSED', 'sum'),
        
        PREV_MEAN_AMT_ANNUITY=('AMT_ANNUITY', 'mean'),
        PREV_MEAN_AMT_APPLICATION=('AMT_APPLICATION', 'mean'),
        PREV_MEAN_AMT_CREDIT=('AMT_CREDIT', 'mean'),
        PREV_MEAN_APPLICATION_CREDIT_RATIO=('APPLICATION_CREDIT_RATIO', 'mean'),
        
        PREV_MEAN_DAYS_FIRST_DUE=('DAYS_FIRST_DUE', 'mean'),
        PREV_MEAN_DAYS_LAST_DUE=('DAYS_LAST_DUE', 'mean'),
        PREV_MEAN_CNT_PAYMENT=('CNT_PAYMENT', 'mean'),
    ).reset_index()
    
    # Key ratios: proportion of applications refused & approved
    agg['PREV_RATIO_REFUSED'] = agg['PREV_COUNT_REFUSED'] / agg['PREV_COUNT_APPLICATIONS']
    agg['PREV_RATIO_APPROVED'] = agg['PREV_COUNT_APPROVED'] / agg['PREV_COUNT_APPLICATIONS']
    
    return agg

prev_app_features = engineer_previous_application_features(prev_app)
print(prev_app_features.shape)
prev_app_features.head()

(338857, 13)


,SK_ID_CURR,PREV_COUNT_APPLICATIONS,PREV_COUNT_APPROVED,PREV_COUNT_REFUSED,PREV_MEAN_AMT_ANNUITY,PREV_MEAN_AMT_APPLICATION,PREV_MEAN_AMT_CREDIT,PREV_MEAN_APPLICATION_CREDIT_RATIO,PREV_MEAN_DAYS_FIRST_DUE,PREV_MEAN_DAYS_LAST_DUE,PREV_MEAN_CNT_PAYMENT,PREV_RATIO_REFUSED,PREV_RATIO_APPROVED
0,100001,1,1,0,3951.000,24835.50,23787.00,0.957782,-1709.000000,-1619.000000,8.0,0.0,1.0
1,100002,1,1,0,9251.775,179055.00,179055.00,1.000000,-565.000000,-25.000000,24.0,0.0,1.0
2,100003,3,3,0,56553.990,435436.50,484191.00,1.057664,-1274.333333,-1054.333333,10.0,0.0,1.0
3,100004,1,1,0,5357.250,24282.00,20106.00,0.828021,-784.000000,-724.000000,4.0,0.0,1.0
4,100005,2,1,0,4813.200,22308.75,20076.75,0.899950,-706.000000,-466.000000,12.0,0.0,0.5


### 2.3 Correlation with Target


In [11]:
df_check = pd.read_csv('../data/processed/application_train_with_anomaly.csv')
df_check = df_check[['SK_ID_CURR', 'TARGET']].merge(prev_app_features, on='SK_ID_CURR', how='left')

correlations = df_check.drop(columns=['SK_ID_CURR']).corr()['TARGET'].sort_values()
print(correlations)

PREV_RATIO_APPROVED                  -0.063521
PREV_MEAN_AMT_ANNUITY                -0.034871
PREV_COUNT_APPROVED                  -0.031553
PREV_MEAN_AMT_APPLICATION            -0.021803
PREV_MEAN_AMT_CREDIT                 -0.016114
PREV_COUNT_APPLICATIONS               0.019762
PREV_MEAN_CNT_PAYMENT                 0.027743
PREV_MEAN_DAYS_LAST_DUE               0.033531
PREV_MEAN_DAYS_FIRST_DUE              0.040066
PREV_COUNT_REFUSED                    0.064469
PREV_MEAN_APPLICATION_CREDIT_RATIO    0.065003
PREV_RATIO_REFUSED                    0.077671
TARGET                                1.000000
Name: TARGET, dtype: float64


**Insight:** `PREV_RATIO_REFUSED` (0.078) is the strongest feature in this table — a history of previous rejections is fairly predictive of current default. Conversely, `PREV_RATIO_APPROVED` correlates negatively (-0.064), which is logically consistent.


### 2.4 Save


In [12]:
prev_app_features.to_csv('../data/processed/prev_app_features.csv', index=False)

## 3. Installments Payments — Installment History

### 3.1 Load & Explore


In [13]:
installments = pd.read_csv('../data/raw/installments_payments.csv')
print(installments.shape)
installments.head()

(13605401, 8)


,SK_ID_PREV,SK_ID_CURR,NUM_INSTALMENT_VERSION,NUM_INSTALMENT_NUMBER,DAYS_INSTALMENT,DAYS_ENTRY_PAYMENT,AMT_INSTALMENT,AMT_PAYMENT
0,1054186,161674,1.0,6,-1180.0,-1187.0,6948.360,6948.360
1,1330831,151639,0.0,34,-2156.0,-2156.0,1716.525,1716.525
2,2085231,193053,2.0,1,-63.0,-63.0,25425.000,25425.000
3,2452527,199697,1.0,3,-2418.0,-2426.0,24350.130,24350.130
4,2714724,167756,1.0,2,-1383.0,-1366.0,2165.040,2160.585


In [14]:
installments.info()
installments.groupby('SK_ID_CURR').size().describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13605401 entries, 0 to 13605400
Data columns (total 8 columns):
 #   Column                  Dtype  
---  ------                  -----  
 0   SK_ID_PREV              int64  
 1   SK_ID_CURR              int64  
 2   NUM_INSTALMENT_VERSION  float64
 3   NUM_INSTALMENT_NUMBER   int64  
 4   DAYS_INSTALMENT         float64
 5   DAYS_ENTRY_PAYMENT      float64
 6   AMT_INSTALMENT          float64
 7   AMT_PAYMENT             float64
dtypes: float64(5), int64(3)
memory usage: 830.4 MB


count    339587.000000
mean         40.064552
std          41.053343
min           1.000000
25%          12.000000
50%          25.000000
75%          51.000000
max         372.000000
dtype: float64

In [15]:
installments.isnull().sum()

SK_ID_PREV                   0
SK_ID_CURR                   0
NUM_INSTALMENT_VERSION       0
NUM_INSTALMENT_NUMBER        0
DAYS_INSTALMENT              0
DAYS_ENTRY_PAYMENT        2905
AMT_INSTALMENT               0
AMT_PAYMENT               2905
dtype: int64

**Insight:** this dataset is large (13.6 million rows), but the only missing values are in `DAYS_ENTRY_PAYMENT` & `AMT_PAYMENT` (2,905 rows) — likely installments that simply haven't been / were never paid. This is handled as `FLAG_NOT_PAID` in feature engineering, not dropped.


### 3.2 Feature Engineering

Main features: lateness (`DAYS_LATE`), payment shortfall (`AMT_SHORTFALL`), and their derived ratios (`INST_RATIO_LATE`, `INST_RATIO_SHORTFALL`, `INST_RATIO_NOT_PAID`).


In [16]:
def engineer_installments_features(installments):
    installments = installments.copy()
    
    # Flag installments that were never paid at all
    installments['FLAG_NOT_PAID'] = installments['AMT_PAYMENT'].isnull().astype(int)
    
    # Compute lateness & shortfall (only valid for installments that have a payment)
    installments['DAYS_LATE'] = installments['DAYS_ENTRY_PAYMENT'] - installments['DAYS_INSTALMENT']
    installments['AMT_SHORTFALL'] = installments['AMT_INSTALMENT'] - installments['AMT_PAYMENT']
    
    # Late flag & underpayment flag
    installments['FLAG_LATE'] = (installments['DAYS_LATE'] > 0).astype(int)
    installments['FLAG_SHORTFALL'] = (installments['AMT_SHORTFALL'] > 0).astype(int)
    
    agg = installments.groupby('SK_ID_CURR').agg(
        INST_COUNT=('SK_ID_PREV', 'count'),
        INST_COUNT_NOT_PAID=('FLAG_NOT_PAID', 'sum'),
        
        INST_MEAN_DAYS_LATE=('DAYS_LATE', 'mean'),
        INST_MAX_DAYS_LATE=('DAYS_LATE', 'max'),
        INST_SUM_FLAG_LATE=('FLAG_LATE', 'sum'),
        
        INST_MEAN_AMT_SHORTFALL=('AMT_SHORTFALL', 'mean'),
        INST_SUM_AMT_SHORTFALL=('AMT_SHORTFALL', 'sum'),
        INST_SUM_FLAG_SHORTFALL=('FLAG_SHORTFALL', 'sum'),
        
        INST_MEAN_AMT_INSTALMENT=('AMT_INSTALMENT', 'mean'),
        INST_MEAN_AMT_PAYMENT=('AMT_PAYMENT', 'mean'),
    ).reset_index()
    
    # Ratio: proportion of installments that were late & that fell short (fairer than raw count)
    agg['INST_RATIO_LATE'] = agg['INST_SUM_FLAG_LATE'] / agg['INST_COUNT']
    agg['INST_RATIO_SHORTFALL'] = agg['INST_SUM_FLAG_SHORTFALL'] / agg['INST_COUNT']
    agg['INST_RATIO_NOT_PAID'] = agg['INST_COUNT_NOT_PAID'] / agg['INST_COUNT']
    
    return agg

installments_features = engineer_installments_features(installments)
print(installments_features.shape)
installments_features.head()

(339587, 14)


,SK_ID_CURR,INST_COUNT,INST_COUNT_NOT_PAID,INST_MEAN_DAYS_LATE,INST_MAX_DAYS_LATE,INST_SUM_FLAG_LATE,INST_MEAN_AMT_SHORTFALL,INST_SUM_AMT_SHORTFALL,INST_SUM_FLAG_SHORTFALL,INST_MEAN_AMT_INSTALMENT,INST_MEAN_AMT_PAYMENT,INST_RATIO_LATE,INST_RATIO_SHORTFALL,INST_RATIO_NOT_PAID
0,100001,7,0,-7.285714,11.0,1,0.0,0.0,0,5885.132143,5885.132143,0.142857,0.0,0.0
1,100002,19,0,-20.421053,-12.0,0,0.0,0.0,0,11559.247105,11559.247105,0.000000,0.0,0.0
2,100003,25,0,-7.160000,-1.0,0,0.0,0.0,0,64754.586000,64754.586000,0.000000,0.0,0.0
3,100004,3,0,-7.666667,-3.0,0,0.0,0.0,0,7096.155000,7096.155000,0.000000,0.0,0.0
4,100005,9,0,-23.555556,1.0,1,0.0,0.0,0,6240.205000,6240.205000,0.111111,0.0,0.0


### 3.3 Correlation with Target


In [17]:
df_check = pd.read_csv('../data/processed/application_train_with_anomaly.csv')
df_check = df_check[['SK_ID_CURR', 'TARGET']].merge(installments_features, on='SK_ID_CURR', how='left')

correlations = df_check.drop(columns=['SK_ID_CURR']).corr()['TARGET'].sort_values()
print(correlations)

INST_MEAN_AMT_PAYMENT      -0.023169
INST_COUNT                 -0.021096
INST_MEAN_AMT_INSTALMENT   -0.018409
INST_MAX_DAYS_LATE          0.004657
INST_RATIO_NOT_PAID         0.013162
INST_COUNT_NOT_PAID         0.017438
INST_MEAN_DAYS_LATE         0.020870
INST_SUM_AMT_SHORTFALL      0.027326
INST_SUM_FLAG_SHORTFALL     0.028300
INST_MEAN_AMT_SHORTFALL     0.029339
INST_SUM_FLAG_LATE          0.030446
INST_RATIO_SHORTFALL        0.062612
INST_RATIO_LATE             0.070015
TARGET                      1.000000
Name: TARGET, dtype: float64


**Insight:** ratios win again — `INST_RATIO_LATE` (0.070) and `INST_RATIO_SHORTFALL` (0.063) are much stronger than their raw counts (`INST_SUM_FLAG_LATE` is only 0.030). This is the third consistent pattern: **ratio > raw count** across all secondary tables so far.


### 3.4 Save


In [18]:
installments_features.to_csv('../data/processed/installments_features.csv', index=False)

## 4. POS_CASH & Credit Card Balance

### 4.1 Load & Explore


In [19]:
pos_cash = pd.read_csv('../data/raw/POS_CASH_balance.csv')
credit_card = pd.read_csv('../data/raw/credit_card_balance.csv')

print("POS_CASH:", pos_cash.shape)
print(pos_cash.head())
print(pos_cash['NAME_CONTRACT_STATUS'].value_counts())

print("\nCREDIT_CARD:", credit_card.shape)
print(credit_card.head())
print(credit_card['NAME_CONTRACT_STATUS'].value_counts())

POS_CASH: (10001358, 8)
   SK_ID_PREV  SK_ID_CURR  MONTHS_BALANCE  CNT_INSTALMENT  \
0     1803195      182943             -31            48.0   
1     1715348      367990             -33            36.0   
2     1784872      397406             -32            12.0   
3     1903291      269225             -35            48.0   
4     2341044      334279             -35            36.0   

   CNT_INSTALMENT_FUTURE NAME_CONTRACT_STATUS  SK_DPD  SK_DPD_DEF  
0                   45.0               Active       0           0  
1                   35.0               Active       0           0  
2                    9.0               Active       0           0  
3                   42.0               Active       0           0  
4                   35.0               Active       0           0  
NAME_CONTRACT_STATUS
Active                   9151119
Completed                 744883
Signed                     87260
Demand                      7065
Returned to the store       5461
Approved       

### 4.2 Feature Engineering

For `credit_card_balance`, the most interesting feature is the **utilization ratio** (`AMT_BALANCE` / `AMT_CREDIT_LIMIT_ACTUAL`) — how much of the credit card limit is being used.


In [20]:
def engineer_pos_cash_features(pos_cash):
    pos_cash = pos_cash.copy()
    
    pos_cash['FLAG_DPD'] = (pos_cash['SK_DPD'] > 0).astype(int)
    
    agg = pos_cash.groupby('SK_ID_CURR').agg(
        POS_COUNT=('SK_ID_PREV', 'count'),
        POS_MEAN_CNT_INSTALMENT_FUTURE=('CNT_INSTALMENT_FUTURE', 'mean'),
        
        POS_MEAN_SK_DPD=('SK_DPD', 'mean'),
        POS_MAX_SK_DPD=('SK_DPD', 'max'),
        POS_SUM_FLAG_DPD=('FLAG_DPD', 'sum'),
        
        POS_MEAN_SK_DPD_DEF=('SK_DPD_DEF', 'mean'),
        POS_MAX_SK_DPD_DEF=('SK_DPD_DEF', 'max'),
    ).reset_index()
    
    agg['POS_RATIO_DPD'] = agg['POS_SUM_FLAG_DPD'] / agg['POS_COUNT']
    
    return agg

pos_cash_features = engineer_pos_cash_features(pos_cash)
print(pos_cash_features.shape)

(337252, 9)


In [21]:
def engineer_credit_card_features(credit_card):
    credit_card = credit_card.copy()
    
    credit_card['FLAG_DPD'] = (credit_card['SK_DPD'] > 0).astype(int)
    
    # Utilization ratio: balance used relative to the available limit
    credit_card['UTILIZATION_RATIO'] = credit_card['AMT_BALANCE'] / credit_card['AMT_CREDIT_LIMIT_ACTUAL']
    credit_card['UTILIZATION_RATIO'] = credit_card['UTILIZATION_RATIO'].replace([np.inf, -np.inf], np.nan)
    
    agg = credit_card.groupby('SK_ID_CURR').agg(
        CC_COUNT=('SK_ID_PREV', 'count'),
        CC_MEAN_AMT_BALANCE=('AMT_BALANCE', 'mean'),
        CC_MEAN_CREDIT_LIMIT=('AMT_CREDIT_LIMIT_ACTUAL', 'mean'),
        CC_MEAN_UTILIZATION=('UTILIZATION_RATIO', 'mean'),
        CC_MAX_UTILIZATION=('UTILIZATION_RATIO', 'max'),
        
        CC_MEAN_DRAWINGS_ATM=('AMT_DRAWINGS_ATM_CURRENT', 'mean'),
        CC_MEAN_DRAWINGS_CURRENT=('AMT_DRAWINGS_CURRENT', 'mean'),
        
        CC_MEAN_SK_DPD=('SK_DPD', 'mean'),
        CC_MAX_SK_DPD=('SK_DPD', 'max'),
        CC_SUM_FLAG_DPD=('FLAG_DPD', 'sum'),
    ).reset_index()
    
    agg['CC_RATIO_DPD'] = agg['CC_SUM_FLAG_DPD'] / agg['CC_COUNT']
    
    return agg

credit_card_features = engineer_credit_card_features(credit_card)
print(credit_card_features.shape)

(103558, 12)


### 4.3 Correlation with Target


In [22]:
df_check = pd.read_csv('../data/processed/application_train_with_anomaly.csv')
df_check = df_check[['SK_ID_CURR', 'TARGET']] \
    .merge(pos_cash_features, on='SK_ID_CURR', how='left') \
    .merge(credit_card_features, on='SK_ID_CURR', how='left')

correlations = df_check.drop(columns=['SK_ID_CURR']).corr()['TARGET'].sort_values()
print(correlations)

CC_COUNT                         -0.060481
POS_COUNT                        -0.035632
CC_SUM_FLAG_DPD                  -0.011331
CC_MEAN_CREDIT_LIMIT             -0.008852
CC_MAX_SK_DPD                    -0.005975
CC_MEAN_SK_DPD                   -0.003195
CC_RATIO_DPD                      0.001753
POS_MAX_SK_DPD                    0.004763
POS_MEAN_SK_DPD                   0.005436
POS_MEAN_SK_DPD_DEF               0.006496
POS_MAX_SK_DPD_DEF                0.009580
POS_SUM_FLAG_DPD                  0.011901
POS_MEAN_CNT_INSTALMENT_FUTURE    0.027827
POS_RATIO_DPD                     0.030616
CC_MEAN_DRAWINGS_CURRENT          0.058732
CC_MEAN_DRAWINGS_ATM              0.059925
CC_MEAN_AMT_BALANCE               0.087177
CC_MAX_UTILIZATION                0.097011
CC_MEAN_UTILIZATION               0.135560
TARGET                            1.000000
Name: TARGET, dtype: float64


**Strongest insight across this whole week's feature engineering:** `CC_MEAN_UTILIZATION` (0.136) — customers who on average use a large portion of their credit card limit are far more likely to default. Its value approaches the level of `EXT_SOURCE_*` from the EDA notebook, making it one of the most valuable features in the entire master table.


### 4.4 Save


In [23]:
pos_cash_features.to_csv('../data/processed/pos_cash_features.csv', index=False)
credit_card_features.to_csv('../data/processed/credit_card_features.csv', index=False)

## 5. Merge Everything into a Master Feature Table


In [24]:
# Load everything that's been saved
df_master = pd.read_csv('../data/processed/application_train_with_anomaly.csv')

bureau_feat = pd.read_csv('../data/processed/bureau_features.csv')
prev_feat = pd.read_csv('../data/processed/prev_app_features.csv')
inst_feat = pd.read_csv('../data/processed/installments_features.csv')
pos_feat = pd.read_csv('../data/processed/pos_cash_features.csv')
cc_feat = pd.read_csv('../data/processed/credit_card_features.csv')

df_master = df_master.merge(bureau_feat, on='SK_ID_CURR', how='left')
df_master = df_master.merge(prev_feat, on='SK_ID_CURR', how='left')
df_master = df_master.merge(inst_feat, on='SK_ID_CURR', how='left')
df_master = df_master.merge(pos_feat, on='SK_ID_CURR', how='left')
df_master = df_master.merge(cc_feat, on='SK_ID_CURR', how='left')

print(df_master.shape)
print(df_master.isnull().sum().sum())

(307511, 149)
3857140


**Insight:** after merging (`left join` into `application_train`), many NaNs appear in the aggregated columns — this isn't actually missing data, it means that customer **has no history** in that table (e.g. never had a credit card). So it's treated differently: filled with 0, not a typical statistical imputation.


In [25]:
def finalize_master_features(df):
    df = df.copy()
    
    # All engineered columns from the 5 secondary tables (consistent prefixes)
    secondary_prefixes = ('BUREAU_', 'PREV_', 'INST_', 'POS_', 'CC_')
    secondary_cols = [c for c in df.columns if c.startswith(secondary_prefixes)]
    
    # Fill all with 0 - NaN here means "no history", not missing data
    df[secondary_cols] = df[secondary_cols].fillna(0)
    
    return df

df_master = finalize_master_features(df_master)
print(df_master.isnull().sum().sum())

0


In [26]:
df_master.to_csv('../data/processed/master_features.csv', index=False)
print(df_master.shape)

(307511, 149)


---
### Week 3 Summary & Next Steps

- 5 secondary tables (`bureau`, `previous_application`, `installments_payments`, `POS_CASH_balance`, `credit_card_balance`) were successfully aggregated into customer-level features, all reusable and ready to be moved to `src/feature_engineering.py`.
- **Strongest methodological insight:** across *all* secondary tables, the **ratio** versions (`BUREAU_RATIO_ACTIVE`, `PREV_RATIO_REFUSED`, `INST_RATIO_LATE`, `CC_MEAN_UTILIZATION`, etc.) consistently outperform raw counts as predictors. This pattern is worth documenting as a standalone finding.
- `CC_MEAN_UTILIZATION` (credit card utilization ratio) is the strongest predictor from this whole process (0.136), approaching the level of `EXT_SOURCE_*`.
- Final master table: **149 columns, 307,511 rows, 0 missing values** — ready for the modeling stage.

**Before moving on to Week 4 (modeling), 3 things still need to be sorted out:**
1. Re-check the full correlation across the master table (top 15–20 strongest features across all tables at once)
2. Check for multicollinearity between features (e.g. `BUREAU_COUNT_ACTIVE` vs `BUREAU_RATIO_ACTIVE`) — important so Model A (Logistic Regression) doesn't run into issues
3. Properly split the train/test data (stratified, since the target is imbalanced)
